# Tutorial 1 - Indicators of Hydrologic Alteration (IHA)

This tutorial demonstrates how to use the `SARAwater` package to compute the Indicators of Hydrologic Alteration (IHA) for a river reach. The IHA metrics help assess the impact of flow alterations on aquatic ecosystems.

## Import libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sarawater as sara

## Import the flow time series

This tutorial needs a CSV file with the flow discharge time series for the reach, with at least two columns: "Date" (the timestamps in a format like YYYY-MM-DD) and "Q" (the flow discharge, in cubic meters per second, m³/s).

You have two options to provide this file:

1. **Use your own data**: place your CSV file in the `data` folder next to this notebook, and update the filename in the cell below.

2. **Use the example dataset**: do nothing: if no local file is found, the notebook automatically downloads the example dataset used in this tutorial from the [SARAwater GitHub repository](https://github.com/sara-acqua/sarawater)

In [ ]:
input_csv_filepath = os.path.join("data", "daily_discharge_7y.csv")

if os.path.isfile(input_csv_filepath):
    print(f"Using your local file: {input_csv_filepath}")
    reach_df = pd.read_csv(input_csv_filepath, parse_dates=["Date"])
else:
    print("No local file found - downloading the example dataset from GitHub instead.")
    GITHUB_DATA_URL = "https://raw.githubusercontent.com/sara-acqua/sarawater/main/tutorials/tutorial_1_IHA/data/daily_discharge_7y.csv"
    reach_df = pd.read_csv(GITHUB_DATA_URL, parse_dates=["Date"])

reach_df.head()

## Initialize a reach object

In [ ]:
# Convert the timestamps of the csv file to a list of datetime objects
datetime_list = reach_df["Date"].dt.to_pydatetime().tolist()

# Put the discharge data into a numpy array
discharge_data = np.array(reach_df["Q"].values)

# Define the maximum flow discharge that can be abstracted from the river reach (in m3/s)
Qabs_max = 3

# Create a Reach object with the imported data
my_reach = sara.Reach("My Reach", datetime_list, discharge_data, Qabs_max)

## Add scenarios to the reach object

### Minimum Flow Requirement (MFR) scenario

In [ ]:
# Define monthly values of minimum release in m3/s
Qreq_months = [
    0.076,
    0.076,
    0.076,
    0.106,
    0.106,
    0.106,
    0.106,
    0.091,
    0.091,
    0.106,
    0.106,
    0.076,
]

# Create a constant scenario with these values
MFR_scenario = sara.ConstScenario(
    name="MFR",
    description="Minimum Flow Requirement scenario from CSV file",
    reach=my_reach,
    Qreq_months=Qreq_months,
)

# Add the scenario to the reach
my_reach.add_scenario(MFR_scenario)

### Ecological scenario (using the built-in method)

In [ ]:
my_reach.add_ecological_flow_scenario(
    "EF", "Ecological Flow Scenario with default parameters"
)

### Proportional release scenario (using the built-in method)

In [ ]:
prop_scenario = sara.PropScenario(
    name="Prop_06_30",
    description="Proportional scenario",
    reach=my_reach,
    Qbase=0.6 * np.min(Qreq_months),
    c_Qin=0.3,
    Qreq_min=np.min(Qreq_months),
    Qreq_max=np.max(my_reach.scenarios[1].Qreq_months),
)
my_reach.add_scenario(prop_scenario)

### Let's check we added the scenarios correctly

In [ ]:
my_reach.print_scenarios()

## How to delete existing scenarios

To delete a scenario, you can run `my_reach.scenarios.pop(N)`, where the number `N` in the parenthesis is the index of the list `scenarios` corresponding to the scenario you want to remove.

In [ ]:
# my_reach.scenarios.pop(2) # This deletes the Proportional Scenario

## Compute the released flow discharge for each scenario and compute the IHA indices (IARI and normalized IHA)

In [ ]:
for scenario in my_reach.scenarios:
    scenario.compute_Qrel()
    scenario.compute_IHA_index(index_metric="IARI")
    scenario.compute_IHA_index(index_metric="normalized_IHA")
    scenario.compute_natural_abstracted_volumes()

Let's take a look at the computed IHA values for a given scenario...

In [ ]:
scenario_to_look_at = my_reach.scenarios[0]

In [ ]:
scenario_to_look_at.IHA.keys()

In [ ]:
scenario_to_look_at.IHA["Group1"].keys()

In [ ]:
scenario_to_look_at.IHA["Group1"]["mean_january"]

...and at the corresponding values of the IARI index, which is also computed for each year and for each group of IHA indices.

In [ ]:
scenario_to_look_at.IARI.aggregated

In [ ]:
scenario_to_look_at.IARI.groups

In [ ]:
scenario_to_look_at.IARI.groups["Group1"]

## Analyse the results using a ReachPlotter object

Set `matplotlib`'s plotting parameters.

In [ ]:
plt.rcParams.update(
    {
        "figure.figsize": (8, 6),
        "font.size": 17,
        "legend.fontsize": 15,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "lines.linewidth": 2.5,
        "mathtext.fontset": "cm",
    }
)

Set color array for the scenarios. The first element of this list will be the color of the first scenario added to the Reach, and so on with the following scenarios.

In [ ]:
scenario_colors = [
    "tab:red",
    "tab:orange",
    "tab:green",
    "tab:purple",
    "tab:brown",
    "tab:pink",
]

### Initialize the plotter

In [ ]:
plotter = sara.ReachPlotter(my_reach, scenario_colors=scenario_colors)

### Plot samples of released flow discharge

In [ ]:
year = 2018
start_date = f"{year}-01-01"
end_date = f"{year}-12-31"
plotter.plot_scenarios_discharge(start_date=start_date, end_date=end_date)
plt.xticks(rotation=45)
plt.tight_layout()

### IHA and IARI plots

In [ ]:
# plotter.plot_iha_parameters()
# plotter.plot_iha_boxplots()
# plotter.plot_iari_groups()
# plotter.plot_iari_summary()
# plotter.plot_cases_duration()
# plotter.plot_monthly_abstraction()
# plotter.plot_nIHA_summary()
# plotter.plot_nIHA_vs_volume()
plotter.plot_iari_vs_volume(save=False)